In [ ]:
#!pip install unsloth trl peft accelerate bitsandbytes datasets

In [2]:
# For GPU check
import torch
print(f"CUDA is install and available to use: {torch.cuda.is_available()}")
print(f"GPU is available to use: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA is install and available to use: True
GPU is available to use: Tesla T4


In [ ]:
from unsloth import FastLanguageModel
import torch

# Same base model already used by the WriteDetect app's AI Judge (llm_utils.py),
# so the fine-tuned adapter we save at the end can drop directly into the app.
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

max_seq_length = 768  # text is capped at ~1200 chars per example, plenty of headroom
dtype = None  # Auto detection

# Load model and tokenizer (0.5B is tiny - no need for 4-bit quantization)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=False,
)

We're fine-tuning the AI Judge from the WriteDetect project: given a piece of text, decide whether it's Human-Written or AI-Generated.

The dataset is `AI_Human.csv` (the Kaggle "AI vs Human Text" dataset) - 487,235 rows with `text` and `generated` columns (`generated` = 1 means AI-written, 0 means human-written). It's heavily imbalanced and far too large to train on in full on a free Colab GPU, so we'll pull a balanced sample and format it into the same prompt structure the WriteDetect app uses at inference time.

In [ ]:
import zipfile
import os
import pandas as pd

# Unzip if you uploaded AI_Human.csv.zip instead of the raw CSV
if os.path.exists("AI_Human.csv.zip") and not os.path.exists("AI_Human.csv"):
    with zipfile.ZipFile("AI_Human.csv.zip", "r") as z:
        z.extractall(".")

N_PER_CLASS = 5000  # 10,000 total - enough for LoRA to learn real patterns, fast on a free T4

human_parts, ai_parts = [], []
human_have, ai_have = 0, 0

for chunk in pd.read_csv("AI_Human.csv", chunksize=200_000):
    if human_have < N_PER_CLASS:
        h = chunk[chunk["generated"] == 0]
        human_parts.append(h)
        human_have += len(h)
    if ai_have < N_PER_CLASS:
        a = chunk[chunk["generated"] == 1]
        ai_parts.append(a)
        ai_have += len(a)
    if human_have >= N_PER_CLASS and ai_have >= N_PER_CLASS:
        break

human_df = pd.concat(human_parts).sample(n=N_PER_CLASS, random_state=42)
ai_df = pd.concat(ai_parts).sample(n=N_PER_CLASS, random_state=42)

df = pd.concat([human_df, ai_df]).sample(frac=1, random_state=42).reset_index(drop=True)
print(df.shape)
print(df["generated"].value_counts().to_dict())

In [ ]:
import random
from datasets import Dataset

# This must match the exact prompt structure used by run_ai_judge() in the
# WriteDetect app's llm_utils.py, so the fine-tuned model responds correctly
# to the same prompts it will see at inference time.
SYSTEM_PROMPT = (
    "You are an expert at detecting whether a piece of writing was written by a "
    "human or generated by an AI language model. Be concise and decisive."
)

USER_TEMPLATE = (
    "Read the text below and decide whether it is Human-Written or AI-Generated.\n"
    "Reply in exactly this format:\n"
    "Verdict: <Human-Written or AI-Generated>\n"
    "Reasoning: <one or two short sentences>\n\n"
    "Text:\n{text}"
)

# We don't have ground-truth reasoning in this dataset, only the label, so we
# sample from several phrasings per class instead of one fixed string - this
# keeps the model from just memorizing a single verbatim sentence.
REASONING_VARIANTS = {
    0: [
        "The phrasing, pacing, and small imperfections in word choice are consistent with how a human typically writes.",
        "There are natural inconsistencies in tone and structure that are typical of human writing rather than machine output.",
        "The sentence flow has the kind of organic variation and minor stylistic quirks usually seen in human-authored text.",
        "The writing shows personal voice and uneven pacing, which is more characteristic of a human author than an AI model.",
    ],
    1: [
        "The text has unusually smooth, evenly structured phrasing and generic word choice typical of AI-generated writing.",
        "The consistent sentence structure and lack of personal voice suggest this was generated by an AI language model.",
        "The writing is fluent but generic, with the kind of uniform structure commonly produced by AI text generators.",
        "The overly polished and repetitive phrasing pattern is characteristic of AI-generated content.",
    ],
}

MAX_CHARS = 1200
random.seed(42)


def build_example(text, label):
    verdict = "Human-Written" if label == 0 else "AI-Generated"
    reasoning = random.choice(REASONING_VARIANTS[label])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(text=str(text)[:MAX_CHARS])},
        {"role": "assistant", "content": f"Verdict: {verdict}\nReasoning: {reasoning}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)


formatted_data = [build_example(row["text"], row["generated"]) for _, row in df.iterrows()]
dataset = Dataset.from_dict({"text": formatted_data})

In [6]:
dataset

Dataset({
    features: ['text'],
    num_rows: 500
})

In [ ]:
# Add LoRA adapters
# r/alpha are lower than the lecture's Phi-3 example (r=64) since Qwen2.5-0.5B
# has far fewer parameters to begin with - r=16 is plenty of capacity here.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Now, lets build a Trainer that can perform the fine-tuning.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Training arguments optimized for Unsloth
# SFTConfig must be used instead of TrainingArguments — SFTTrainer requires it
# for checkpointing to work. dataset_text_field, max_seq_length, and
# dataset_num_proc also move into SFTConfig (deprecated as direct SFTTrainer args)
#
# Batch size is higher than the lecture's Phi-3 example since Qwen2.5-0.5B is
# much smaller (0.5B vs 3.8B params) and there's more headroom on the T4.
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,  # Effective batch size = 16
        warmup_steps=20,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none",  # Disable Weights & Biases logging
    ),
)

In [9]:
# Train the model
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 3 | Total steps = 189
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,0.453414
50,0.150686
75,0.134791
100,0.122880
125,0.116181
150,0.112548
175,0.111026


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-63/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-63.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-126/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-126.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-189/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-189.


Now, let's test if the fine-tuned model actually works. The test prompt must use the same chat-template structure (system + user turns) as `run_ai_judge()` in `llm_utils.py`, since that's what the model was trained on.

In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

test_text = (
    "The implementation of renewable energy infrastructure represents a critical "
    "pathway toward achieving global sustainability targets. Furthermore, it is "
    "essential to consider the multifaceted economic implications associated with "
    "this transition, as well as the technological innovations that facilitate it."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_TEMPLATE.format(text=test_text)},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=100,
    use_cache=True,
    temperature=0.3,
    do_sample=False,
)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

Save and Download Model

Unlike the lecture's GGUF export (for Ollama/llama.cpp), we save a standard PEFT adapter here, since the WriteDetect app loads the AI Judge directly via `transformers` + `peft.PeftModel`, not via GGUF/llama.cpp.

In [ ]:
import shutil

OUTPUT_DIR = "qwen_judge_lora"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

shutil.make_archive("qwen_judge_lora", "zip", OUTPUT_DIR)
print(f"Saved adapter to {OUTPUT_DIR}/ and zipped it to qwen_judge_lora.zip")
print(
    "Download the zip, unzip it, and copy its contents (NOT any checkpoint-* "
    "subfolders) into <WriteDetect project>/models/qwen_judge_lora/ to replace "
    "the existing adapter."
)

In [ ]:
from google.colab import files

files.download("qwen_judge_lora.zip")

References:

1. https://www.youtube.com/watch?v=pTaSDVz0gok

2. https://www.youtube.com/watch?v=eC6Hd1hFvos&t=657s

3. Dataset: Kaggle "AI Vs Human Text" (`AI_Human.csv`, 487,235 rows, `text`/`generated` columns)

4. Target project: WriteDetect's AI Judge (`llm_utils.py`, `models/qwen_judge_lora/`)